In [5]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

# 1. Load Data (assuming you downloaded application_train.csv from Kaggle)
print("Loading data...")
df = pd.read_csv('../home-credit-default-risk-dataset/application_train.csv')

# For this MVP, we will only use a few key columns to keep it simple
features = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'CODE_GENDER']
target = 'TARGET'

X = df[features].copy()
y = df[target]

# 2. Preprocessing
print("Cleaning data...")
# Convert categorical text ('CODE_GENDER') into binary/dummy columns (1s and 0s)
X = pd.get_dummies(X, columns=['CODE_GENDER'], drop_first=True)

# Split data: 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Impute missing numeric values with the median
imputer = SimpleImputer(strategy='median')
X_train_clean = imputer.fit_transform(X_train)
X_test_clean = imputer.transform(X_test)

# 3. Model Training
print("Training LightGBM model...")
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    class_weight='balanced' # Crucial: Tells the model to care about the minority class (defaults)
)
model.fit(X_train_clean, y_train)

# 4. Evaluation
print("Evaluating model...")
# We use predict_proba to get the *probability* of default (class 1), not just a hard 0 or 1
y_pred_proba = model.predict_proba(X_test_clean)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"MVP Model ROC-AUC Score: {roc_auc:.4f}")

# Save the trained model and the imputer for deployment
# joblib.dump(model, 'lgbm_model.pkl')
# joblib.dump(imputer, 'imputer.pkl')
# print("Model saved successfully.")


Loading data...
Cleaning data...
Training LightGBM model...
[LightGBM] [Info] Number of positive: 19876, number of negative: 226132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002166 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1177
[LightGBM] [Info] Number of data points in the train set: 246008, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
Evaluating model...
MVP Model ROC-AUC Score: 0.6497
